# SPECTER2: first embeddings

Embed a handful of known papers and check that the similarities make sense before touching the corpus.

In [1]:
import pandas as pd
import torch
from adapters import AutoAdapterModel
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer

## Load the model

SPECTER2 is the `specter2_base` model plus the `specter2` (proximity) adapter, loaded with the `adapters` library rather than sentence-transformers.

In [2]:
tokenizer = AutoTokenizer.from_pretrained("allenai/specter2_base")
model = AutoAdapterModel.from_pretrained("allenai/specter2_base")
model.load_adapter("allenai/specter2", source="hf", load_as="specter2", set_active=True)

print(model.active_adapters)  # expect Stack[specter2]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Stack[specter2]


In [3]:
def embed(papers):
    """Return one 768-number vector per paper: the [CLS] token of title [SEP] abstract."""
    texts = [p["title"] + tokenizer.sep_token + (p.get("abstract") or "") for p in papers]
    inputs = tokenizer(texts, padding=True, truncation=True,
                       return_tensors="pt", return_token_type_ids=False, max_length=512)
    with torch.no_grad():  # inference only, so skip the bookkeeping needed for training
        output = model(**inputs)
    return output.last_hidden_state[:, 0, :].numpy()


def similarity_table(embeds, names):
    return pd.DataFrame(cosine_similarity(embeds), index=names, columns=names).round(3)

## Test papers

Chosen to separate topic from method: INLA SDM papers, INLA papers outside SDMs, a non-INLA SDM paper, and something unrelated.

In [4]:
papers = [{'title': 'Spatial species distribution models: Using Bayes inference with INLA and SPDE to improve the tree species choice for important European tree species', 'abstract': 'Species distribution models (SDMs) are a standard tool for predicting species occurrence under climate change. Despite its limitations, SDMs have been widely used to assist forest management decisions in their choice of future tree species. The accuracy of SDMs is often affected by heterogeneous occurrence data caused by different forest inventory schemes, historical processes, climate and insect calamities and more. These processes bias the relationships modelled between species occurrence and climate. However, numerous studies have shown that explicit modelling of spatial effects may improve model accuracy. In this study, we applied the Integrated Nested Laplace Approximation (INLA) and Stochastic Partial Differential Equations (SPDE) algorithms as a means to accomplish Bayes inference for Generalized Additive Models (GAMs) with explicit modelling of spatial effects. Our results show that including spatial effects in GAM-based SDMs improved model performance and accuracy leading to more reliable predictions for the species Favorability. Conditional predictions that remove spatial effects from the models allow us to distinguish core and marginal ecological ranges with less bias through forest management, which may support the tree-species choice for climate-resilient forests.'},
          {'title': 'Integrated species distribution models fitted in INLA are sensitive to mesh parameterisation', 'abstract': ' The ever-growing popularity of citizen science, as well as recent technological and digital developments, have allowed the collection of data on species distributions at an extraordinary rate. In order to take advantage of these data, information of varying quantity and quality needs to be integrated. Point process models have been proposed as an elegant way to achieve this for estimates of species distributions. These models can be fitted efficiently using Bayesian methods based on integrated nested Laplace approximations (INLA) with stochastic partial differential equations (SPDEs). This approach uses an efficient way to model spatial autocorrelation using a Gaussian random field and a triangular mesh over the spatial domain. The mesh is constructed by user-defined variables, so effectively represents a free parameter in the model. However, there is a lack of understanding about how to set these mesh parameters, and their effect on model performance. Here, we assess how mesh parameters affect predictions and model fit to estimate the distribution of the serotine bat, Eptesicus serotinus, in Great Britain. A Bayesian INLA model was fitted using five meshes of varying densities to a dataset comprising both structured observations from a national monitoring programme and opportunistic records. We demonstrate that mesh density impacted spatial predictions with a general loss of accuracy with increasing mesh coarseness. However, we also show that the finest mesh was unable to overcome spatial biases in the data. In addition, the magnitude of the covariate effects differed markedly between meshes. This confirms that mesh parameterisation is an important and delicate process with implications for model inference. We discuss how species distribution modellers might adapt their use of INLA in the light of these findings.'},
          {'title': 'Spatio-temporal data integration for species distribution modelling in R-INLA', 'abstract': 'Species distribution modelling is a highly used tool for understanding and predicting biodiversity change, and recent work has emphasised the importance of understanding how species distributions change over both time and space. Spatio-temporal models require large amounts of data spread over time and space, and as such are clear candidates to benefit from model-based integration of different data sources. However, spatio-temporal models are highly computationally intensive and integrating different data sources can make this approach even more unfeasible to ecologists. Here we demonstrate how the R-INLA methodology can be used for model-based data integration for spatio-temporally explicit modelling of species distribution change. We demonstrate that this method can be applied to both point and areal data with two contrasting case studies, one using the SPDE approach for modelling spatio-temporal change in the Gatekeeper butterfly (Pyronia tithonus) across Great Britain and the second using a spatio-temporal areal model to describe change in caddisfly (Trichoptera) populations across the River Thames catchment. We show that in the caddisfly case study integrating together different data sources led to greater understanding of the change in abundance across the River Thames both seasonally and over 5 years of data. However, in the butterfly case study moving to a spatio-temporal context exacerbated differences between the data sources and resulted in no greater ecological insight into change in the Gatekeeper population. Our work provides a computationally feasible framework for spatio-temporally explicit integration of data within SDMs and demonstrates both the potential benefits and the challenges in applying this methodology to real ecological data.'},
          {'title': 'Spatio-temporal disease mapping using INLA', 'abstract': 'Spatio-temporal disease mapping models are a popular tool to describe the pattern of disease counts. They are usually formulated in a hierarchical Bayesian framework with latent Gaussian model. So far, computationally expensive Markov chain Monte Carlo algorithms have been used for parameter estimation which might induce a large Monte Carlo error. An alternative method using integrated nested Laplace approximations (INLA) has recently been proposed. A major advantage of INLA is that it returns accurate parameter estimates in short computational time. Additionally, the deviance information criterion is provided for Bayesian model choice. This paper describes how several parametric and nonparametric models and extensions thereof can be fitted to space–time count data using INLA. Particular emphasis is given to the appropriate choice of linear constraints to ensure identifiability of the parameter estimates. The models are applied to counts of Salmonellosis in cattle reported to the Swiss Federal Veterinary Office 1991–2008. Copyright © 2010 John Wiley & Sons, Ltd.'},
          {'title': 'Direct fitting of dynamic models using integrated nested Laplace approximations — INLA', 'abstract': 'Inference in state-space models usually relies on recursive forms for filtering and smoothing of the state vectors regarding the temporal structure of the observations, an assumption that is, from our view point, unnecessary if the dataset is fixed, that is, completely available before analysis. In this paper, we propose a computational framework to perform approximate full Bayesian inference in linear and generalized dynamic linear models based on the Integrated Nested Laplace Approximation (INLA) approach. The proposed framework directly approximates the posterior marginals of interest disregarding the assumption of recursive updating/estimation of the states and hyperparameters in the case of fixed datasets and, therefore, enable us to do fully Bayesian analysis of complex state-space models more easily and in a short computational time. The proposed framework overcomes some limitations of current tools in the dynamic modeling literature and is vastly illustrated with a series of simulated as well as well known real-life examples from the literature, including realistically complex models with correlated error structures and models with more than one state vector, being mutually dependent on each other. R code is available online for all the examples presented.'},
          {'title': 'Efficacy of species distribution models (SDMs) for ecological realms to ascertain biological conservation and practices', 'abstract': 'SDMs are not new to conservation, but their popularity has increased dramatically in recent years. This step-by-step review provides an overview of the efficacy of SDMs in guiding restoration and conservation strategies across a wide range of ecological realms. Numerous studies have demonstrated the applicability of SDMs to various fields; however, their effectiveness has not been evaluated for a variety of ecosystems. Therefore, a survey and analysis of published work on the use of SDM in ecological rejuvenation and conservation from 2002 to 2023 (May) is conducted. The analysis found a total of 739 papers and the number of papers increased after 2016. The United States of America (135) had the most SDM implementations in conservation planning, followed by China (59), Australia (40), and other nations, according to the classification of the research area by country. In the model, Maxent (341) and in the areas, Forest (252) outperformed contenders for the number of papers published. This review will create a framework to aid in the following: (1) information about taxa and realms in need of protection, (2) selection of the best SDM approach according to study aim, focused species, and study area, and (3) supplemental techniques useful for better SDM output. In addition, it will discuss the advantages and disadvantages of various fundamental SDM algorithms in the context of ecological conservation.'},
          {'title': 'Impacts of bat use of anthropogenic structures on bats and humans', 'abstract': 'Human-induced landscape modifications and climate change are forcing wildlife into closer contact with humans as the availability of natural habitats decreases. Although the importance of anthropogenic structures for the conservation of species is widely recognized, negative narratives surrounding bats may impede conservation efforts in human-dominated landscapes. We conducted a global systematic literature review to summarize research pertaining to bats in anthropogenic structures and analyze the impacts of occupancy of these structures on bats and humans. We extracted data from 735 publications and included 8 that provided a total of 29 quantitative estimates in meta-analyses assessing the consequences of roost selection by bats in anthropogenic and natural habitats. Additionally, information from all 735 publications was used for summaries. Research focused on the Northern Hemisphere, despite the highest diversity of bat species occurring near the equator. Of the 13 identified impacts on bats from the use of anthropogenic structures, disturbance (caused by, e.g., visitation, renovations, artificial lighting) was the most frequently reported. Effects of bat presence on humans were primarily associated with pathogens or other microorganisms of zoonotic interest. Buildings were the most frequently identified anthropogenic roost, and the use of buildings differed across biogeographic realms. Although impacts varied across realms and structures, the Nearctic and Palearctic had the highest incidence of impacts. Few studies compared anthropogenic roosts with natural roosts, but our meta-analyses broadly identified differences in the effects of artificial versus natural roosts on bat behavior, roost temperature, and bat health and occupancy. We found that research is not focused currently on areas where bat–human interactions are most likely to intensify with the growing rate of urbanization. Although many effects on bats from roosting in anthropogenic structures were documented or mentioned, most studies did not measure these effects and few compared them with natural roosts. Quantifying impacts could help in the design of management practices that would benefit bats and humans.'}]

names = ["INLA trees", "INLA ISDM mesh", "INLA spatio-temporal SDM", "INLA disease mapping",
         "INLA dynamic models", "SDM review (no INLA)", "Bats"]
assert len(names) == len(papers)

## Similarities with the adapter

In [5]:
embeds = embed(papers)
print(embeds.shape)  # expect (7, 768)
similarity_table(embeds, names)

(7, 768)


,INLA trees,INLA ISDM mesh,INLA spatio-temporal SDM,INLA disease mapping,INLA dynamic models,SDM review (no INLA),Bats
INLA trees,1.000,0.921,0.902,0.893,0.901,0.931,0.867
INLA ISDM mesh,0.921,1.000,0.927,0.911,0.916,0.891,0.887
INLA spatio-temporal SDM,0.902,0.927,1.000,0.893,0.880,0.890,0.855
INLA disease mapping,0.893,0.911,0.893,1.000,0.927,0.856,0.865
INLA dynamic models,0.901,0.916,0.880,0.927,1.000,0.869,0.856
SDM review (no INLA),0.931,0.891,0.890,0.856,0.869,1.000,0.886
Bats,0.867,0.887,0.855,0.865,0.856,0.886,1.000


## Experiment: base model without the adapter

Switches the adapter off, embeds again, then switches it back on so the rest of the notebook is unaffected.

In [6]:
model.set_active_adapters(None)
base_embeds = embed(papers)
model.set_active_adapters("specter2")
print(model.active_adapters)  # back to Stack[specter2]

similarity_table(base_embeds, names)

There are adapters available but none are activated for the forward pass.
There are adapters available but none are activated for the forward pass.


Stack[specter2]


,INLA trees,INLA ISDM mesh,INLA spatio-temporal SDM,INLA disease mapping,INLA dynamic models,SDM review (no INLA),Bats
INLA trees,1.000,0.934,0.915,0.908,0.920,0.949,0.882
INLA ISDM mesh,0.934,1.000,0.938,0.938,0.925,0.918,0.912
INLA spatio-temporal SDM,0.915,0.938,1.000,0.913,0.886,0.919,0.882
INLA disease mapping,0.908,0.938,0.913,1.000,0.931,0.879,0.877
INLA dynamic models,0.920,0.925,0.886,0.931,1.000,0.873,0.855
SDM review (no INLA),0.949,0.918,0.919,0.879,0.873,1.000,0.893
Bats,0.882,0.912,0.882,0.877,0.855,0.893,1.000


## Findings

- **Ordering is sensible; scale is compressed.** The bats paper is least similar to everything, but scores sit in 0.85–0.93. Judge by rank, not raw score: a score means nothing without the other scores for the same query.
- **Topic and method compete.** "INLA trees" is nearest the non-INLA SDM review (0.931), while "INLA disease mapping" groups with the other INLA papers. SPECTER2 sees both "SDM" and "INLA"; which wins depends on the abstract. For the review, "SDMs *using INLA*" is an intersection embeddings will blur, so keyword matching on INLA (hybrid search, Stage 4) will matter. First entry for the evaluation set.
- **The adapter spreads scores apart.** Without it almost everything scores higher (bats vs ISDM mesh 0.912 vs 0.887) and related and unrelated pairs are harder to separate. It also ranks the INLA spatio-temporal SDM above disease mapping for the ISDM mesh paper, where the base model ties them. Consistent with SPECTER2's citation-based training.
- **The load-time warning is spurious.** The embeddings change when the adapter is switched off, so it was active.